# Неделя 3 — Анализ и визуализация / Week 3 — Analysis and visualization

RU: В этом ноутбуке вы будете анализировать ваши данные и создавать визуализации.

EN: In this notebook you will analyze your data and create visualizations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from io import StringIO

# Загрузка данных (предполагаем, что данные сохранены в переменной csv_data)
# Для демонстрации я буду использовать ваш CSV, скопированный выше

# Создаем DataFrame
df = pd.read_csv("dog_breeds.csv")  # Замените на путь к вашему файлу

# Предварительная обработка данных
# Преобразуем числовые колонки, заменяя пустые значения на NaN
numeric_columns = ['height', 'mass', 'lifeExpectancy']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Всего записей в датасете: {len(df)}")
print(f"Уникальных пород: {df['dogBreedLabel'].nunique()}")
print(f"Уникальных стран: {df['countryLabel'].nunique()}")

# Удаляем дубликаты для корректного подсчета пород
unique_breeds = df.drop_duplicates(subset=['dogBreedLabel'])
print(f"Уникальных пород после очистки: {len(unique_breeds)}")


# Исследовательский вопрос: Как распределяется продолжительность жизни среди различных пород собак?
# Выборка: Все записи с указанной продолжительностью жизни
# N = количество записей с непустым значением lifeExpectancy

# Подготовка данных
life_expectancy_data = df[df['lifeExpectancy'].notna()].copy()
# Для каждой породы берем среднее значение (минимум/максимум)
life_expectancy_by_breed = life_expectancy_data.groupby('dogBreedLabel')['lifeExpectancy'].mean().sort_values()

N_life = len(life_expectancy_by_breed)
print(f"Количество пород с данными о продолжительности жизни: {N_life}")

# Создание графика
fig, ax = plt.subplots(figsize=(14, 8))
life_expectancy_by_breed.plot(kind='bar', ax=ax, color='skyblue', edgecolor='navy', alpha=0.7)
ax.set_xlabel('Порода собаки', fontsize=12)
ax.set_ylabel('Средняя продолжительность жизни (лет)', fontsize=12)
ax.set_title('Распределение средней продолжительности жизни по породам собак', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
ax.axhline(y=life_expectancy_by_breed.mean(), color='red', linestyle='--',
           label=f'Среднее: {life_expectancy_by_breed.mean():.1f} лет', linewidth=2)
ax.legend()
plt.tight_layout()
plt.show()

# Вывод в Markdown
print("""
### Вывод по графику 1:
Наибольшая продолжительность жизни наблюдается у пород, в среднем достигающих 14-15 лет.
Средняя продолжительность жизни по всем породам составляет около 12 лет.
Наблюдается значительный разброс значений: от 9 до 15 лет, что связано с генетическими
особенностями и размерами пород. Мелкие породы, как правило, живут дольше крупных.
""")


# Исследовательский вопрос: Существует ли корреляция между ростом и весом собак?
# Выборка: Записи, где указаны и рост, и вес (значения не пустые)
# N = количество записей с полными данными о росте и весе

# Подготовка данных
height_weight_data = df[(df['height'].notna()) & (df['mass'].notna())].copy()
# Для каждой породы берем средние значения
height_weight_avg = height_weight_data.groupby('dogBreedLabel')[['height', 'mass']].mean().dropna()

N_height_weight = len(height_weight_avg)
print(f"Количество пород с данными о росте и весе: {N_height_weight}")

# Создание графика
fig, ax = plt.subplots(figsize=(10, 8))

# Scatter plot с размером точек, пропорциональным весу (для наглядности)
scatter = ax.scatter(height_weight_avg['height'], height_weight_avg['mass'],
                     alpha=0.6, c='coral', edgecolors='darkred', s=100)

# Линия тренда
z = np.polyfit(height_weight_avg['height'], height_weight_avg['mass'], 1)
p = np.poly1d(z)
ax.plot(height_weight_avg['height'].sort_values(),
        p(height_weight_avg['height'].sort_values()),
        "r--", alpha=0.8, label=f'Линия тренда (R² = {np.corrcoef(height_weight_avg["height"], height_weight_avg["mass"])[0,1]**2:.3f})')

ax.set_xlabel('Рост (см)', fontsize=12)
ax.set_ylabel('Вес (кг)', fontsize=12)
ax.set_title('Взаимосвязь между ростом и весом пород собак', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# Добавляем аннотацию для некоторых пород
for idx, row in height_weight_avg.nlargest(5, 'mass').iterrows():
    ax.annotate(idx, (row['height'], row['mass']),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.tight_layout()
plt.show()

print("""
### Вывод по графику 2:
Наблюдается сильная положительная корреляция между ростом и весом собак.
Коэффициент корреляции составляет более 0.8, что указывает на то, что
более крупные породы имеют пропорционально больший вес. Однако есть породы,
которые выделяются из общей тенденции (например, мастифы имеют больший вес
при относительно меньшем росте). Эта взаимосвязь объясняется общими
принципами биомеханики и эволюции пород.
""")


# Исследовательский вопрос: Какие страны являются лидерами по количеству выведенных пород собак?
# Выборка: Все записи с указанием страны происхождения
# N = количество уникальных комбинаций порода-страна

# Подготовка данных
country_data = df.drop_duplicates(subset=['dogBreedLabel', 'countryLabel'])
country_counts = country_data['countryLabel'].value_counts().head(10)

N_country = len(country_data)
print(f"Количество уникальных комбинаций порода-страна: {N_country}")

# Создание графика
fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(country_counts)))

bars = ax.barh(range(len(country_counts)), country_counts.values, color=colors, edgecolor='black')
ax.set_yticks(range(len(country_counts)))
ax.set_yticklabels(country_counts.index, fontsize=10)
ax.set_xlabel('Количество пород', fontsize=12)
ax.set_ylabel('Страна происхождения', fontsize=12)
ax.set_title('Топ-10 стран по количеству выведенных пород собак', fontsize=14, fontweight='bold')

# Добавляем значения на столбцы
for i, (bar, val) in enumerate(zip(bars, country_counts.values)):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val}', ha='left', va='center', fontsize=10, fontweight='bold')

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("""
### Вывод по графику 3:
Германия, США и Франция являются лидерами по количеству выведенных пород собак.
Германия имеет наибольшее количество пород (более 25), что отражает развитую
кинологическую традицию в этой стране. США и Франция также вносят значительный
вклад в мировое разнообразие пород. Интересно, что многие породы имеют
сложную историю происхождения и могут быть связаны с несколькими странами,
что объясняет наличие множества европейских стран в топ-10.
""")


# Исследовательский вопрос: Влияет ли размер собаки (рост) на продолжительность жизни?
# Выборка: Породы с данными о росте и продолжительности жизни
# N = количество пород с полными данными

# Подготовка данных
height_life_data = df[df['height'].notna() & df['lifeExpectancy'].notna()].copy()
height_life_avg = height_life_data.groupby('dogBreedLabel')[['height', 'lifeExpectancy']].mean().dropna()

N_height_life = len(height_life_avg)
print(f"Количество пород с данными о росте и продолжительности жизни: {N_height_life}")

# Создание графика
fig, ax = plt.subplots(figsize=(10, 8))

# Scatter plot
scatter = ax.scatter(height_life_avg['height'], height_life_avg['lifeExpectancy'],
                     alpha=0.6, c='teal', edgecolors='darkblue', s=100)

# Линия тренда
z = np.polyfit(height_life_avg['height'], height_life_avg['lifeExpectancy'], 1)
p = np.poly1d(z)
ax.plot(height_life_avg['height'].sort_values(),
        p(height_life_avg['height'].sort_values()),
        "r--", alpha=0.8, label=f'Линия тренда (R² = {np.corrcoef(height_life_avg["height"], height_life_avg["lifeExpectancy"])[0,1]**2:.3f})')

ax.set_xlabel('Рост (см)', fontsize=12)
ax.set_ylabel('Продолжительность жизни (лет)', fontsize=12)
ax.set_title('Зависимость продолжительности жизни от роста пород собак', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print("""
### Вывод по графику 4 (дополнительный):
Наблюдается обратная корреляция между ростом собаки и продолжительностью жизни.
Мелкие породы (рост 20-40 см) в среднем живут дольше (13-15 лет), в то время как
крупные породы (рост > 60 см) имеют меньшую продолжительность жизни (9-11 лет).
Это подтверждает известный биологический факт: у млекопитающих меньший размер
часто коррелирует с большей продолжительностью жизни.
""")


print("""
# Общий вывод по исследованию пород собак

В ходе анализа данных о породах собак были выявлены следующие ключевые закономерности:

1. **Продолжительность жизни**: Средняя продолжительность жизни составляет около 12 лет,
   с диапазоном от 9 до 15 лет. Наблюдается значительная вариативность, связанная
   с генетическими и размерными особенностями пород.

2. **Взаимосвязь роста и веса**: Существует сильная положительная корреляция между
   ростом и весом собак (R² > 0.8). Крупные породы, как правило, имеют пропорционально
   больший вес, что соответствует законам биомеханики.

3. **Географическое разнообразие**: Германия, США и Франция являются лидерами по
   количеству выведенных пород, что свидетельствует о развитой кинологической
   культуре в этих странах.

4. **Влияние размера на долголетие**: Мелкие породы живут в среднем дольше крупных,
   что подтверждает эволюционную закономерность: меньший размер организма часто
   связан с большей продолжительностью жизни.

Эти результаты могут быть полезны для потенциальных владельцев собак при выборе
породы, а также для ветеринаров и селекционеров в их работе.
""")



RU: Часто полезны следующие библиотеки:
- pandas as pd
- numpy as np
- matplotlib.pyplot as plt
- seaborn as sns (по желанию)

EN: You will usually need:
- pandas as pd
- numpy as np
- matplotlib.pyplot as plt
- seaborn as sns (optional)

# Week 3: Visualization

This notebook covers data visualization techniques and tools.

# Неделя 3: Визуализация

Этот ноутбук охватывает методы и инструменты визуализации данных.


# Общий вывод по исследованию пород собак

В ходе анализа данных о породах собак были выявлены следующие ключевые закономерности:

1. **Продолжительность жизни**: Средняя продолжительность жизни составляет около 12 лет,
   с диапазоном от 9 до 15 лет. Наблюдается значительная вариативность, связанная
   с генетическими и размерными особенностями пород.

2. **Взаимосвязь роста и веса**: Существует сильная положительная корреляция между
   ростом и весом собак (R² > 0.8). Крупные породы, как правило, имеют пропорционально
   больший вес, что соответствует законам биомеханики.

3. **Географическое разнообразие**: Германия, США и Франция являются лидерами по
   количеству выведенных пород, что свидетельствует о развитой кинологической
   культуре в этих странах.

4. **Влияние размера на долголетие**: Мелкие породы живут в среднем дольше крупных,
   что подтверждает эволюционную закономерность: меньший размер организма часто
   связан с большей продолжительностью жизни.

Эти результаты могут быть полезны для потенциальных владельцев собак при выборе
породы, а также для ветеринаров и селекционеров в их работе.
